<a href="https://colab.research.google.com/github/mufaddalkc/northstar-ai-employee-knowledge-explorer/blob/main/Mufaddal's_Skill_Builder_Lab_Context_Engineering_and_Vector_Databases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Build an AI Employee Knowledge Explorer

## Business Scenario

Northstar Systems stores employee guidance across separate HR, IT, Benefits, Finance, Onboarding, and Security documents. Employees often ask questions in their own words, which may differ from the policy wording. Northstar wants a **searchable employee knowledge explorer** that finds the most relevant information and shows its source.

You will build a guided retrieval system that:
1. Loads the supplied fictional knowledge-base documents with source and department metadata.
2. Creates a **TF-IDF + cosine similarity** baseline to represent lexical retrieval.
3. Generates **dense embeddings** and builds a **FAISS IndexFlatIP** semantic search index with stable record IDs.
4. Uses **LlamaIndex Documents, Nodes, and an IngestionPipeline** for retrieval-ready content.
5. Stores nodes and metadata in **persistent ChromaDB**, searches them, applies metadata filters, and updates a record.
6. Compares retrieval results and records the limitations of each method.

**Scope:** Google Colab; beginner–intermediate. This assignment focuses on *retrieval and context quality*, not building a web interface or training an ML model. An LLM-based answer is an optional extension.

## Learning Outcomes

By completing this assignment, you should be able to:

- Explain why context relevance, provenance, and token budget matter.
- Implement a TF-IDF baseline and use cosine similarity to rank documents.
- Generate sentence embeddings with `BAAI/bge-small-en-v1.5`.
- Build and query a normalized-vector FAISS index using `IndexFlatIP` and `IndexIDMap`.
- Convert document text and metadata into LlamaIndex `Document` and `Node` objects.
- Store/search retrieval-ready nodes in a persistent ChromaDB collection.
- Use metadata filtering, record updates, and basic retrieval evaluation.
- Identify lexical retrieval failures, cross-method differences, and unsupported questions.

## Provided Resources

Download and keep the supplied [Northstar_Employee_Knowledge_Base.zip](https://drive.google.com/file/d/13QUv3jdmur6Cjvff9IQZm3BXtnDYFYoz/view?usp=drive_link) file. It contains six fictional UTF-8 text files:

| File | Department/category |
|---|---|
| `hr_leave_policy.txt` | HR |
| `it_device_support.txt` | IT |
| `benefits_and_wellbeing.txt` | Benefits |
| `finance_expense_reimbursement.txt` | Finance |
| `people_onboarding.txt` | Onboarding |
| `security_data_protection.txt` | Security |

The dataset is fictional and is for training only. Fixed test questions are provided near the end. **Do not substitute different documents** when preparing the final submission, so that results can be evaluated consistently.

## Submission

Upload your completed `.ipynb` with **executed outputs** to a GitHub repository and share the repository link. Include the six provided knowledge files or the supplied ZIP in your repository. Do **not** upload downloaded embedding-model caches, Chroma database files, API keys, or other large generated files.

## Part A — Prepare the Colab Environment and Knowledge Base
### Step 0. Install libraries and upload the supplied ZIP

Run the two setup cells below. These cells are provided so that students can focus on the 12 unfinished functions. The embedding-model download requires internet access in Colab; the required retrieval activities do **not** require an LLM API key.

- Upload the **exact supplied ZIP**, not a folder of other documents.
- The setup extracts only `.txt` files into `/content/northstar_knowledge_base`.
- Use the fixed category mapping below so search results can be compared fairly.
- If a fresh Colab runtime reports a library import error after installation, restart the runtime and rerun the setup cells.

**Purpose of Code:** Install the packages used in the three lessons, load required imports and model settings, then make the six policy documents available for later retrieval and vector-store operations.

In [1]:
!pip -q install -U scikit-learn sentence-transformers faiss-cpu chromadb llama-index-core

from pathlib import Path
import io
import zipfile
import time
import numpy as np
import pandas as pd
import faiss
import chromadb
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from llama_index.core import Document
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

MODEL_NAME = "BAAI/bge-small-en-v1.5"
CATEGORY_BY_FILE = {
    "hr_leave_policy.txt": "HR",
    "it_device_support.txt": "IT",
    "benefits_and_wellbeing.txt": "Benefits",
    "finance_expense_reimbursement.txt": "Finance",
    "people_onboarding.txt": "Onboarding",
    "security_data_protection.txt": "Security",
}

DATA_DIR = Path("/content/northstar_knowledge_base")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Ready to receive:", len(CATEGORY_BY_FILE), "knowledge documents")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
Ready to receive: 6 knowledge documents


In [2]:
uploaded_zip = files.upload()
filename = "Northstar_Employee_Knowledge_Base.zip"
if filename not in uploaded_zip:
    raise ValueError(f"Please upload the supplied file: {filename}")

with zipfile.ZipFile(io.BytesIO(uploaded_zip[filename])) as archive:
    for member in archive.namelist():
        name = Path(member).name
        if name in CATEGORY_BY_FILE:
            (DATA_DIR / name).write_bytes(archive.read(member))

missing = [name for name in CATEGORY_BY_FILE if not (DATA_DIR / name).exists()]
if missing:
    raise ValueError(f"Some required files are missing from the ZIP: {missing}")
print("Extracted:", ", ".join(sorted(CATEGORY_BY_FILE)))

Saving Northstar_Employee_Knowledge_Base.zip to Northstar_Employee_Knowledge_Base.zip
Extracted: benefits_and_wellbeing.txt, finance_expense_reimbursement.txt, hr_leave_policy.txt, it_device_support.txt, people_onboarding.txt, security_data_protection.txt


### Step 1. Read the employee knowledge documents

**Your task:** Complete `load_knowledge_files(data_dir)`.

- Read the six `.txt` files in the order of `CATEGORY_BY_FILE` (Python dictionaries preserve insertion order).
- For each file, load its UTF-8 text and trim surrounding whitespace.
- Create a dictionary with **four keys**: `id`, `source`, `category`, and `text`.
- Assign stable integer IDs starting at `1001` (`1001`, `1002`, …).
- Return a list containing one dictionary per non-empty document.

**Expected output:** Six document dictionaries, for example `{'id': 1001, 'source': 'hr_leave_policy.txt', 'category': 'HR', 'text': '...'}`.

**Hint:** Use `Path.read_text(encoding='utf-8')`, `enumerate(CATEGORY_BY_FILE, start=1001)`, and `CATEGORY_BY_FILE[filename]`.

**Purpose of Code:** Turn raw policy files into structured retrieval units with stable IDs and provenance metadata. Later search functions will use these fields to show which document supplied each result.

In [3]:
def load_knowledge_files(data_dir: Path) -> list[dict]:
    """Return knowledge records with id, source, category, and text."""
    records = []
    # TODO 1: Loop through the filenames in CATEGORY_BY_FILE.
    for doc_id, filename in enumerate(CATEGORY_BY_FILE, start=1001):
    # TODO 2: Read and clean each text file.
      file_path = data_dir / filename
      cleaned_text = file_path.read_text(encoding='utf-8').strip()
    # TODO 3: Store the text with its stable ID and metadata.
      if cleaned_text:
        records.append({
            'id': doc_id,
            'source': filename,
            'category': CATEGORY_BY_FILE[filename],
            'text': cleaned_text
        })
    # TODO 4: Return the list of document dictionaries.
    return records

## Part B — Context Engineering and a TF-IDF Baseline
### Step 2. Build a TF-IDF representation

**Your task:** Complete `build_tfidf_index(units)`.

- Extract document text from the structured records.
- Create a `TfidfVectorizer(stop_words='english')`.
- Fit it on the documents and obtain the document-term matrix.
- Return `(vectorizer, tfidf_matrix)`.

**Expected output:** A fitted vectorizer and a matrix with one row per document.

**Hint:** Recall the lexical retrieval baseline from *Introduction to Context Engineering*.

**Purpose of Code:** Create a simple retrieval baseline that scores documents using overlapping vocabulary before introducing dense semantic embeddings.

In [5]:
def build_tfidf_index(units: list[dict]):
    """Return (fitted TfidfVectorizer, document TF-IDF matrix)."""
    # TODO 1: Extract a list of text strings from units.
    texts = [unit['text'] for unit in units]
    # TODO 2: Create the vectorizer and fit it on those strings.
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(texts)
    # TODO 3: Return the vectorizer and the transformed matrix.
    return vectorizer, tfidf_matrix

### Step 3. Retrieve documents using TF-IDF and cosine similarity

**Your task:** Complete `tfidf_search(question, units, vectorizer, tfidf_matrix, top_k=3)`.

- Transform the question using the **same fitted vectorizer**.
- Compute cosine similarity between the question vector and document matrix.
- Rank records from highest score to lowest; return up to `top_k` matches.
- Each returned dictionary must include `id`, `source`, `category`, `text`, and a floating-point `score`.
- **Ignore zero-score documents.** If the question has no lexical match, return `[]`.

**Example output:** `[{'source': 'it_device_support.txt', ..., 'score': 0.29}, ...]` (score shown only as an illustration).

**Hint:** Use `cosine_similarity(q_vector, tfidf_matrix)[0]` and `np.argsort(scores)[::-1]`.

**Purpose of Code:** Rank the policy documents by shared terms and expose when keyword-based retrieval has no signal. This gives a baseline to compare with FAISS and ChromaDB.

In [8]:
def tfidf_search(
    question: str,
    units: list[dict],
    vectorizer,
    tfidf_matrix,
    top_k: int = 3,
) -> list[dict]:
    """Return ranked nonzero TF-IDF matches with source metadata."""
    # TODO 1: Transform the question into a TF-IDF vector.
    q_vector = vectorizer.transform([question])
    # TODO 2: Calculate cosine similarity against all documents.
    scores = cosine_similarity(q_vector, tfidf_matrix)[0]
    # TODO 3: Rank matches, ignore zero scores, and limit to top_k.
    ranked_indices = np.argsort(scores)[::-1]

    results = []
    for idx in ranked_indices:
      score = float(scores[idx])
      if score <= 0:
        continue
      unit = units[idx]
      results.append({
          'id': unit['id'],
          'source': unit['source'],
          'category': unit['category'],
          'text': unit['text'],
          'score': score
      })
      if len(results) == top_k:
        break
    # TODO 4: Return matching records with float scores.
    return results

### Step 4. Assemble a readable context block

**Your task:** Complete `format_context(results, max_chars=1600)`.

- Turn a list of retrieved matches into a single readable text block.
- Prefix each passage with a **source label** such as `[Source: it_device_support.txt | IT]`.
- Include the matching document text; stay within the total `max_chars` character budget.
- If no passages are provided, return **exactly** `Insufficient context.`.

**Expected output:** A source-aware context string ready to be placed in an LLM prompt (even though LLM generation is optional here).

**Hint:** Use `result['source']`, `result['category']`, `result['text']`, and `len()` to keep the output within the budget.

**Purpose of Code:** Build relevant, traceable, and length-controlled context. This demonstrates context engineering beyond retrieving a list of passages.

In [14]:
def format_context(results: list[dict], max_chars: int = 1600) -> str:
    """Format search results as a source-labelled, character-limited context."""
    # TODO 1: Return 'Insufficient context.' if results is empty.
    if not results:
      return 'Insufficient context.'
    # TODO 2: Add a readable source label before each passage.
    # TODO 3: Respect the maximum length while joining the passages.
    formatted_chunks = []
    current_length = 0
    separator = "\n\n"

    for res in results:
      label = f"[Source: {res['source']} | {res['category']}]"
      block = f"{label}\n{res['text'].strip()}"

      extra_len = len(separator) if formatted_chunks else 0

      if current_length + extra_len + len(block) <= max_chars:
        formatted_chunks.append(block)
        current_length += extra_len + len(block)
      else:
        if not formatted_chunks and max_chars > 0:
          formatted_chunks.append(block[:max_chars])
        break
    # TODO 4: Return the final context string.
    return separator.join(formatted_chunks)

## Part C — Embeddings and FAISS Semantic Search
### Step 5. Generate dense document embeddings

**Your task:** Complete `embed_documents(units, model)`.

- Encode all document texts with the supplied `SentenceTransformer` model.
- Use `convert_to_numpy=True` and `normalize_embeddings=True`.
- Convert the result to a contiguous, two-dimensional NumPy array of `float32` values.
- Return the embedding matrix; do not create a vector index in this step.

**Expected output:** A matrix with six rows and one embedding vector per document.

**Hint:** See the `model.encode(...)` usage in *Introduction to FAISS*.

**Purpose of Code:** Convert document meaning into dense numerical vectors so that semantically related questions can match policies even when their wording is different.

In [17]:
def embed_documents(units: list[dict], model: SentenceTransformer) -> np.ndarray:
    """Return normalized float32 embeddings, one row per knowledge unit."""
    # TODO 1: Collect document texts from units.
    texts = [unit['text'] for unit in units]
    # TODO 2: Encode with the supplied embedding model and normalization.
    raw_embeddings = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    # TODO 3: Convert to contiguous float32 NumPy vectors.
    vectors = np.ascontiguousarray(raw_embeddings, dtype=np.float32)
    if vectors.ndim == 1:
      vectors = vectors.reshape(-1, 1)
    # TODO 4: Return the embedding matrix.
    return vectors

### Step 6. Build a FAISS vector index with stable IDs

**Your task:** Complete `build_faiss_index(vectors, units)`.

- Create an `IndexFlatIP` using the embedding dimension.
- Wrap it in `faiss.IndexIDMap`.
- Add the normalized vectors with their stable integer `id` values using `add_with_ids`.
- Create a lookup dictionary mapping `id -> original document record`.
- Return `(index, units_by_id)`.

**Expected output:** A FAISS index containing six records and a dictionary that restores source text/metadata from search IDs.

**Hint:** Use `vectors.shape[1]`, `np.asarray(..., dtype=np.int64)`, and `faiss.IndexIDMap(faiss.IndexFlatIP(dimension))`.

**Purpose of Code:** Store normalized semantic vectors in a searchable FAISS index and preserve stable IDs so returned matches can be connected to their original policy documents.

In [20]:
def build_faiss_index(vectors: np.ndarray, units: list[dict]):
    """Return (FAISS IndexIDMap, dictionary from integer IDs to records)."""
    # TODO 1: Read the embedding dimension from vectors.
    dimension = vectors.shape[1]
    # TODO 2: Construct IndexFlatIP and wrap it with IndexIDMap.
    index = faiss.IndexIDMap(faiss.IndexFlatIP(dimension))
    # TODO 3: Add the vectors using the stable record IDs.
    ids = np.asarray([unit['id'] for unit in units], dtype=np.int64)
    index.add_with_ids(vectors, ids)
    # TODO 4: Build the ID-to-document lookup and return both objects.
    units_by_id = {unit['id']: unit for unit in units}
    return index, units_by_id

### Step 7. Search with FAISS

**Your task:** Complete `faiss_search(question, model, index, units_by_id, top_k=3)`.

- Encode and normalize the question using the **same embedding model**.
- Search the FAISS index for up to `top_k` nearest results.
- Convert valid returned IDs back into the original records.
- Return dictionaries with `id`, `source`, `category`, `text`, and a floating-point `score`.
- Skip FAISS's `-1` ID if encountered.

**Expected output:** Ranked semantic document matches with source labels.

**Hint:** `index.search(query_vector, k)` returns `(scores, ids)`; scores from normalized `IndexFlatIP` are cosine-style similarities.

**Purpose of Code:** Retrieve policy documents based on semantic similarity, including questions that do not share exact vocabulary with their relevant documents.

In [24]:
def faiss_search(
    question: str,
    model: SentenceTransformer,
    index,
    units_by_id: dict,
    top_k: int = 3,
) -> list[dict]:
    """Return ranked FAISS matches with original text and metadata."""
    # TODO 1: Encode and normalize the question to float32.
    raw_query = model.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    query_vector = np.ascontiguousarray(raw_query, dtype=np.float32)
    # TODO 2: Search the supplied index for the nearest IDs.
    scores, ids = index.search(query_vector, top_k)
    # TODO 3: Convert valid IDs to original records using units_by_id.
    # TODO 4: Include numeric similarity scores in returned matches.
    results = []
    for score, doc_id in zip(scores[0], ids[0]):
      if int(doc_id) == -1:
        continue
      unit = units_by_id[int(doc_id)]
      results.append({
          'id': unit['id'],
          'source': unit['source'],
          'category': unit['category'],
          'text': unit['text'],
          'score': float(score)
      })

    return results

## Part D — LlamaIndex Documents and Nodes
### Step 8. Convert records into LlamaIndex Documents

**Your task:** Complete `create_llama_documents(units)`.

- Construct one `Document(text=..., metadata=...)` for each structured policy record.
- Each document must retain `source`, `category`, and `record_id` metadata.
- Return the list of LlamaIndex documents.

**Expected output:** Six `Document` objects with source-aware metadata.

**Hint:** This matches the document abstraction used in *Advanced Vector Databases with ChromaDB & LlamaIndex*.

**Purpose of Code:** Represent the policies as LlamaIndex `Document` objects while preserving the provenance needed for filtering, source display, and later updates.

In [27]:
def create_llama_documents(units: list[dict]) -> list[Document]:
    """Return LlamaIndex Documents with source, category, and record_id."""
    # TODO 1: Iterate over the original policy records.
    # TODO 2: Create Document(text=..., metadata=...) for each.
    # TODO 3: Return all created Document objects.
    documents = []
    for unit in units:
      doc = Document(
          text=unit['text'],
          metadata={
              'source': unit['source'],
              'category': unit['category'],
              'record_id': unit['id']
          }
      )
      documents.append(doc)
    return documents

### Step 9. Build an ingestion pipeline and create Nodes

**Your task:** Complete `create_nodes(documents)`.

- Create `SentenceSplitter(chunk_size=180, chunk_overlap=20)`.
- Create an `IngestionPipeline(transformations=[splitter])`.
- Run the pipeline on `documents` and return the resulting nodes.
- Do not remove `source` or `category` metadata from the nodes.

**Expected output:** A non-empty list of smaller retrieval-ready Nodes.

**Hint:** `pipeline.run(documents=documents)` was demonstrated in Lesson 3.

**Purpose of Code:** Divide larger documents into LlamaIndex Nodes so ChromaDB can retrieve focused passages rather than always returning an entire policy file.

In [30]:
def create_nodes(documents: list[Document]) -> list:
    """Return retrieval-ready Nodes using a SentenceSplitter pipeline."""
    # TODO 1: Make a SentenceSplitter with the required settings.
    splitter = SentenceSplitter(chunk_size=180, chunk_overlap=20)
    # TODO 2: Make an IngestionPipeline with this transformation.
    pipeline = IngestionPipeline(transformations=[splitter])
    # TODO 3: Run the pipeline on the Documents and return nodes.
    nodes = pipeline.run(documents=documents)
    return nodes

## Part E — ChromaDB Storage, Metadata Search, and Updates
### Step 10. Save LlamaIndex Nodes in persistent ChromaDB

**Your task:** Complete `upsert_nodes_to_chroma(nodes, model, collection)`.

- Get each Node's text via `node.get_content()` and keep its metadata.
- Generate one **normalized `float32` embedding** for each node using the same model as FAISS.
- Create deterministic IDs in input order: `node-0000`, `node-0001`, etc. (this fixed lab dataset makes the order stable).
- Use `collection.upsert(ids=..., documents=..., metadatas=..., embeddings=...)`.
- Return the number of Nodes upserted.

**Expected output:** The number of retrieval-ready records stored in the collection. Running the same dataset twice should **update**, not duplicate, records.

**Hint:** Chroma accepts embedding arrays as lists; use `vectors.tolist()`. Metadata must contain `source` and `category` from each Node.

**Purpose of Code:** Persist LlamaIndex Nodes together with their embeddings and metadata, allowing semantic queries, metadata filters, and record updates in ChromaDB.

In [33]:
def upsert_nodes_to_chroma(nodes: list, model: SentenceTransformer, collection) -> int:
    """Upsert Nodes (text, embeddings, metadata) and return stored count."""
    if not nodes:
      return 0
    # TODO 1: Collect texts, deterministic IDs, and source/category metadata.
    ids = []
    texts = []
    metadatas = []
    for idx, node in enumerate(nodes):
        ids.append(f"node-{idx:04d}")
        texts.append(node.get_content())
        meta = {
            "source": str(node.metadata["source"]),
            "category": str(node.metadata["category"]),
        }
        if "record_id" in node.metadata:
            meta["record_id"] = int(node.metadata["record_id"])
        metadatas.append(meta)
    # TODO 2: Encode the Node texts into normalized float32 vectors.
    raw_embeddings = model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    vectors = np.ascontiguousarray(raw_embeddings, dtype=np.float32)
    # TODO 3: Call collection.upsert with aligned IDs, documents,
    #         metadatas, and embeddings.
    collection.upsert(ids=ids, documents=texts, metadatas=metadatas, embeddings=vectors.tolist())
    # TODO 4: Return the number of Nodes processed.
    return len(nodes)

### Step 11. Search ChromaDB with an optional metadata filter

**Your task:** Complete `chroma_search(question, model, collection, top_k=3, category=None)`.

- Encode the question with the **same normalized embedding model**.
- Use `collection.query` with the query embedding and `n_results`.
- When `category` is supplied (for example, `'IT'`), apply `where={'category': category}`.
- Extract matching `documents`, `metadatas`, `ids`, and `distances`.
- Return dictionaries with `id`, `source`, `category`, `text`, and `distance`.
- If the collection is empty, return `[]`.

**Important:** Chroma returns **distances** (smaller is better), whereas normalized FAISS `IndexFlatIP` returns **similarity scores** (larger is better). Do not compare those raw values directly.

**Purpose of Code:** Retrieve semantically relevant LlamaIndex Nodes from persistent ChromaDB, optionally narrowing search to a particular policy department before ranking.

In [41]:
def chroma_search(
    question: str,
    model: SentenceTransformer,
    collection,
    top_k: int = 3,
    category: str | None = None,
) -> list[dict]:
    """Return Chroma matches with document text, metadata, and distance."""
    # TODO 1: Return [] for an empty collection.
    total = collection.count()
    if total == 0:
        return []

    # TODO 2: Create the normalized query embedding.
    raw_query = model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    query_vector = np.ascontiguousarray(raw_query, dtype=np.float32)

    # TODO 3: Query ChromaDB, with a category filter when provided.
    query_kwargs = {
        "query_embeddings": query_vector.tolist(),
        "n_results": min(top_k, total),
        "include": ["documents", "metadatas", "distances"],
    }
    if category is not None:
        query_kwargs["where"] = {"category": category}

    raw = collection.query(**query_kwargs)

    # TODO 4: Convert the nested Chroma result to a list of dictionaries.
    ids = raw.get("ids", [[]])[0]
    docs = raw.get("documents", [[]])[0]
    metas = raw.get("metadatas", [[]])[0]
    dists = raw.get("distances", [[]])[0]

    results = []
    for doc_id, text, meta, dist in zip(ids, docs, metas, dists):
        results.append({
            "id": doc_id,
            "source": meta["source"],
            "category": meta["category"],
            "text": text,
            "distance": float(dist),
        })

    return results

### Step 12. Update ChromaDB record metadata

**Your task:** Complete `update_record_status(collection, record_id, status)`.

- Read the Chroma record identified by `record_id` using `collection.get`.
- If no such ID is present, raise `ValueError`.
- Keep its existing metadata and add/update `review_status` with the provided `status`.
- Call `collection.update` to save the new metadata.
- Read the updated record again and return its metadata dictionary.

**Expected output:** Metadata that includes the original `source`, `category`, and the requested `review_status`.

**Hint:** Use `.get(ids=[...], include=['metadatas'])` and `.update(ids=[...], metadatas=[...])`.

**Purpose of Code:** Demonstrate how a managed vector collection supports record updates without discarding provenance or recreating the entire database.

In [42]:
def update_record_status(collection, record_id: str, status: str) -> dict:
    """Update the review_status metadata of an existing Chroma record."""
    # TODO 1: Read the record and validate that it exists.
    existing = collection.get(ids=[record_id], include=["metadatas"])
    if not existing.get("ids") or not existing.get("metadatas"):
        raise ValueError(f"Record '{record_id}' not found in collection.")
    # TODO 2: Preserve old metadata and set review_status.
    updated_metadata = dict(existing["metadatas"][0] or {})
    updated_metadata["review_status"] = status
    # TODO 3: Update the record in ChromaDB.
    collection.update(ids=[record_id], metadatas=[updated_metadata])
    # TODO 4: Re-read and return its updated metadata.
    refreshed = collection.get(ids=[record_id], include=["metadatas"])
    if not refreshed.get("metadatas"):
        raise ValueError(f"Record '{record_id}' missing after update.")
    return dict(refreshed["metadatas"][0])

## Part F — Connect the Components and Test Retrieval
### Step 13. Run the integration checks

**Your task:** Run the provided cell once all 12 functions are implemented. Resolve any errors before continuing.

The cell checks document loading, TF-IDF, normalized embeddings, FAISS indexing, LlamaIndex Nodes, Chroma persistence, department filtering, context formatting, and metadata updates. These are **structural checks**, not a guarantee that every search result is correct.

**Purpose of Code:** Verify that the individual student-written functions work together as one retrieval pipeline and preserve the document-to-metadata links throughout the workflow.

In [43]:
# PROVIDED TEST HARNESS — Execute only after completing Steps 1–12.
units = load_knowledge_files(DATA_DIR)
assert len(units) == 6, "Expected exactly six knowledge files."
assert len({u["id"] for u in units}) == 6, "Every document needs a unique ID."
assert all({"id", "source", "category", "text"} <= set(u) for u in units)
assert all(isinstance(u["text"], str) and u["text"].strip() for u in units)
print("Document loading:", len(units), "records")

vectorizer, tfidf_matrix = build_tfidf_index(units)
assert tfidf_matrix.shape[0] == len(units)
lexical = tfidf_search("How do I report a broken laptop?", units, vectorizer, tfidf_matrix)
assert isinstance(lexical, list)
print("TF-IDF first source:", lexical[0]["source"] if lexical else "No match")

model = SentenceTransformer(MODEL_NAME)
vectors = embed_documents(units, model)
assert isinstance(vectors, np.ndarray) and vectors.ndim == 2
assert vectors.shape[0] == len(units) and vectors.dtype == np.float32
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-3)
index, units_by_id = build_faiss_index(vectors, units)
assert index.ntotal == len(units)
semantic = faiss_search("My work computer stopped working", model, index, units_by_id)
assert isinstance(semantic, list) and semantic
print("FAISS first source:", semantic[0]["source"])

llama_documents = create_llama_documents(units)
assert len(llama_documents) == len(units)
assert all("source" in doc.metadata and "category" in doc.metadata for doc in llama_documents)
nodes = create_nodes(llama_documents)
assert len(nodes) >= len(llama_documents)
assert all("source" in node.metadata and "category" in node.metadata for node in nodes)
print("LlamaIndex nodes:", len(nodes))

chroma_client = chromadb.PersistentClient(path="/content/northstar_chroma_data")
collection = chroma_client.get_or_create_collection(name="northstar_employee_knowledge")
stored_count = upsert_nodes_to_chroma(nodes, model, collection)
assert stored_count == len(nodes)
assert collection.count() >= len(nodes)
print("Chroma stored records:", collection.count())

filtered = chroma_search("My laptop is damaged", model, collection, top_k=3, category="IT")
assert filtered and all(r["category"] == "IT" for r in filtered)
print("Filtered result source:", filtered[0]["source"])

sample_context = format_context(semantic[:2], max_chars=600)
assert isinstance(sample_context, str) and len(sample_context) <= 600
assert "Source:" in sample_context
assert format_context([]) == "Insufficient context."
print("Context preview:\n", sample_context[:350])

sample_id = collection.get(limit=1)["ids"][0]
updated = update_record_status(collection, sample_id, "approved")
assert updated["review_status"] == "approved"
assert "source" in updated and "category" in updated
print("Updated record metadata:", updated)
print("All structural checks passed.")

Document loading: 6 records
TF-IDF first source: security_data_protection.txt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FAISS first source: it_device_support.txt
LlamaIndex nodes: 12
Chroma stored records: 12
Filtered result source: it_device_support.txt
Context preview:
 [Source: it_device_support.txt | IT]
NORTHSTAR SYSTEMS | IT DEVICES AND ACCOUNT SUPPORT GUIDE
Department: IT
Policy owner: IT Service Desk

Broken or malfunctioning company laptop: If your work computer fails to start, overheats, has a damaged screen, or stops working, create an IT Service Desk ticket under Hardware > Repair or Replacement. Include
Updated record metadata: {'source': 'hr_leave_policy.txt', 'record_id': 1001, 'review_status': 'approved', 'category': 'HR'}
All structural checks passed.


### Step 14. Compare lexical and semantic retrieval on fixed questions

**Your task:** Run the provided comparison cell and inspect the results.

- For every question, inspect the first returned **source file**, not just the score.
- The `Expected source` column is an instructor-provided reference, **not** a guarantee that an algorithm will rank it first.
- Examine at least **two questions using different wording** from the documents.
- For the out-of-scope question, describe why a high-ranked result can still be irrelevant.
- A Chroma result is a **Node**, whereas TF-IDF and FAISS search the **whole text file**. Compare source relevance carefully; these are not identical retrieval granularities.

**Purpose of Code:** Evaluate whether retrieval returns the relevant source documents and understand how different representations and retrieval units affect search behavior.

In [44]:
TEST_CASES = [
    ("My work computer stopped working. Who arranges a replacement?", "it_device_support.txt"),
    ("Can I take time off for an urgent family matter?", "hr_leave_policy.txt"),
    ("Can I get money back for a cab to a client meeting?", "finance_expense_reimbursement.txt"),
    ("Where do I learn about the company health cover?", "benefits_and_wellbeing.txt"),
    ("What should I finish before my first working day?", "people_onboarding.txt"),
    ("I accidentally clicked a suspicious link. What now?", "security_data_protection.txt"),
    ("What does the cafeteria serve next Tuesday?", None),  # Out of scope
]

rows = []
for question, expected in TEST_CASES:
    lexical_hits = tfidf_search(question, units, vectorizer, tfidf_matrix, top_k=3)
    faiss_hits = faiss_search(question, model, index, units_by_id, top_k=3)
    chroma_hits = chroma_search(question, model, collection, top_k=3)
    first_source = lambda results: results[0]["source"] if results else "No match"
    rows.append({
        "Question": question,
        "Expected source": expected or "Not in documents",
        "TF-IDF top source": first_source(lexical_hits),
        "FAISS top source": first_source(faiss_hits),
        "Chroma top source": first_source(chroma_hits),
        "TF-IDF source hit": expected is not None and first_source(lexical_hits) == expected,
        "FAISS source hit": expected is not None and first_source(faiss_hits) == expected,
        "Chroma source hit": expected is not None and first_source(chroma_hits) == expected,
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)
known_questions = comparison_df[comparison_df["Expected source"] != "Not in documents"]
for column in ["TF-IDF source hit", "FAISS source hit", "Chroma source hit"]:
    print(f"{column}: {known_questions[column].sum()} / {len(known_questions)}")

print("\nIT-only Chroma results:")
for item in chroma_search("I need help with an account password", model, collection, category="IT"):
    print(item["source"], item["category"], item["text"][:100])

,Question,Expected source,TF-IDF top source,FAISS top source,Chroma top source,TF-IDF source hit,FAISS source hit,Chroma source hit
0,My work computer stopped working. Who arranges...,it_device_support.txt,it_device_support.txt,it_device_support.txt,it_device_support.txt,True,True,True
1,Can I take time off for an urgent family matter?,hr_leave_policy.txt,hr_leave_policy.txt,hr_leave_policy.txt,hr_leave_policy.txt,True,True,True
2,Can I get money back for a cab to a client mee...,finance_expense_reimbursement.txt,finance_expense_reimbursement.txt,finance_expense_reimbursement.txt,finance_expense_reimbursement.txt,True,True,True
3,Where do I learn about the company health cover?,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,benefits_and_wellbeing.txt,True,True,True
4,What should I finish before my first working day?,people_onboarding.txt,hr_leave_policy.txt,people_onboarding.txt,hr_leave_policy.txt,False,True,False
5,I accidentally clicked a suspicious link. What...,security_data_protection.txt,security_data_protection.txt,security_data_protection.txt,security_data_protection.txt,True,True,True
6,What does the cafeteria serve next Tuesday?,Not in documents,benefits_and_wellbeing.txt,hr_leave_policy.txt,hr_leave_policy.txt,False,False,False


TF-IDF source hit: 5 / 6
FAISS source hit: 6 / 6
Chroma source hit: 5 / 6

IT-only Chroma results:
it_device_support.txt IT NORTHSTAR SYSTEMS | IT DEVICES AND ACCOUNT SUPPORT GUIDE
Department: IT
Policy owner: IT Service Des
it_device_support.txt IT The IT team may verify your identity before restoring access.

New software: Employees must request 


### Step 15. Write your observations (Markdown response)

Add a **new Markdown cell** after this one and answer in your own words:

1. For which two questions did TF-IDF and FAISS differ? Which retrieved source was more useful, and why?
2. How did the ChromaDB `category='IT'` filter change the results?
3. Why might comparing raw FAISS scores with raw Chroma distances be misleading?
4. Explain why the out-of-scope cafeteria question must not be treated as answered, even if the vector store retrieves something.
5. What are two ways that metadata and context limits improve the usefulness or safety of an LLM's final answer?

**Purpose of this step:** Interpret the search outputs rather than assuming that receiving a result means it is correct or sufficiently grounded.

### Observations & Retrieval Analysis

1. **Where TF-IDF and FAISS differed:**
   - **Question 4 (*"What should I finish before my first working day?"*):** TF-IDF returned `hr_leave_policy.txt` due to surface-level keyword overlap (such as "working" or "day"), whereas FAISS correctly retrieved `people_onboarding.txt` because dense embeddings captured the semantic intent around pre-joining onboarding tasks. FAISS's result was far more useful here.
   - **Question 6 (*"What does the cafeteria serve next Tuesday?"*):** TF-IDF returned `benefits_and_wellbeing.txt` while FAISS returned `hr_leave_policy.txt`. Neither source is useful because cafeteria menus do not exist anywhere in the Northstar knowledge base.

2. **Effect of the ChromaDB `category='IT'` filter:**
   - Applying `where={'category': 'IT'}` restricted the search space strictly to IT department nodes before similarity ranking. As a result, all returned passages came exclusively from `it_device_support.txt` rather than pulling password or security snippets from other departments like Security (`security_data_protection.txt`).

3. **Why comparing raw FAISS scores with raw Chroma distances is misleading:**
   - Normalized FAISS `IndexFlatIP` computes **inner product / cosine similarity**, where **higher values** (closer to `1.0`) indicate stronger semantic similarity. In contrast, ChromaDB returns **distance metrics** (such as squared L2 distance), where **smaller values** (closer to `0.0`) indicate closer matches. Because they operate on opposite scales and metrics, comparing their raw numbers directly is invalid.

4. **Why the out-of-scope cafeteria question must not be treated as answered:**
   - Nearest-neighbor vector search always returns the closest available documents in the index, even when a query is completely unrelated to the stored knowledge base. Treating a high-ranked neighbor as valid evidence for an unsupported topic would cause an LLM to hallucinate or give misleading policy advice instead of refusing with *"Insufficient context."*

5. **Two ways metadata and context limits improve LLM usefulness and safety:**
   - **Source Provenance & Filtering (Metadata):** Attaching `source`, `category`, and `review_status` lets the system restrict searches to approved, department-specific policies and cite exact source documents so employees can verify answers.
   - **Noise Reduction & Budget Control (Context Limits):** Enforcing a character budget (`max_chars`) keeps prompts within model context limits and prevents low-relevance trailing chunks from diluting the most relevant policy passages.

## Optional Extension — Generate a Grounded Answer with DeepSeek

This is **not required for core marks**. If you have time and already know how to use an LLM API from GenAI Foundations, you may build a short answer generator.

**Your task:** Complete the optional function below using the API pattern from the previous module.

- Retrieve at most two Chroma matches and format them with `format_context`.
- Use a system instruction telling the LLM to answer **only from the provided context**, identify the source, and say `Insufficient context.` when evidence is missing.
- Read `DEEPSEEK_API_KEY` securely from Colab Secrets; never print or commit it.
- Treat retrieved results as **possible evidence**, not proof of an answer. Test the out-of-scope question explicitly.

**Note:** Vector search always tries to find near neighbors, even for an unrelated question. A prompt alone cannot guarantee reliable refusal; discuss how stronger relevance checks could help.

**Purpose of Code:** Connect your improved retrieval pipeline to the LLM generation techniques from GenAI Foundations without making API access a requirement of this module.

In [ ]:
# OPTIONAL: requires a DeepSeek API key and additional request setup from the earlier chapter.
def generate_grounded_answer(question: str, retrieved_results: list[dict]) -> str:
    """Return an LLM answer using only source-labelled retrieved context."""
    # TODO 1: Format at most two retrieved passages using format_context.
    # TODO 2: Prepare a grounded system/user prompt and source instructions.
    # TODO 3: Call the LLM API using your earlier chapter's secure setup.
    # TODO 4: Return the generated text (or an insufficient-context response).
    pass

## Evaluation Rubric

| Component | Marks |
|---|---:|
| Load documents with stable IDs and metadata | 10 |
| TF-IDF ranking and context formatting | 20 |
| Normalized dense embeddings and FAISS retrieval | 25 |
| LlamaIndex Documents and Nodes | 15 |
| ChromaDB storage, filtering, and metadata update | 20 |
| Testing, interpretation, and notebook clarity | 10 |
| **Total** | **100** |

The optional LLM extension is for extra practice, not part of the required 100 marks. Evaluate *correctness and meaningful interpretation*, not whether every retrieval method always returns the expected source first.

## Submission Checklist

- [ ] I implemented **all 12 required functions** in Steps 1–12.
- [ ] I used the **six supplied Northstar policy files** and kept their source/category metadata.
- [ ] My Colab notebook runs the integration checks and displays the retrieval comparison table.
- [ ] I tested the ChromaDB category filter and updated record metadata.
- [ ] I explained retrieval differences and the out-of-scope question in a Markdown cell.
- [ ] I uploaded the **executed notebook** and the supplied data ZIP/files to GitHub.
- [ ] I did not include API keys, ChromaDB generated folders, or model-cache directories in my repository.

**Reminder:** Leave all generated outputs visible in your submitted notebook.  